#Введение и предобработка

Общий заказчик для обеих частей задания - страховая компания, занимающаяся страхованием жизни и транспорта.

---
В нашей части работы мы будем смотреть [датасет](https://www.kaggle.com/datasets/steverusso/cincinnati-car-crash-data) официальных данных по автомобильным авариям в 2010-2021 годах в городах Цинцинати и Огайо. Данные собраны с официального портала штатов. Наша задача - предсказывать летальность аварии.

---

In [24]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import datetime as dt

In [9]:
df = pd.read_csv("cincinnati_traffic_crash_data__cpd.csv").drop(columns = 'Unnamed: 0')
df.head()

/tmp/ipykernel_2633/3656284645.py:1: DtypeWarning: Columns (25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("cincinnati_traffic_crash_data__cpd.csv").drop(columns = 'Unnamed: 0')


,ADDRESS_X,LATITUDE_X,LONGITUDE_X,AGE,COMMUNITY_COUNCIL_NEIGHBORHOOD,CPD_NEIGHBORHOOD,CRASHDATE,CRASHLOCATION,CRASHSEVERITY,CRASHSEVERITYID,...,LOCALREPORTNO,MANNEROFCRASH,ROADCONDITIONSPRIMARY,ROADCONTOUR,ROADSURFACE,SNA_NEIGHBORHOOD,TYPEOFPERSON,WEATHER,ZIP,UNITTYPE
0,63XX GRACELY,39.107808,-84.688195,31-40,SAYLER PARK,SAYLER PARK,06/17/2014 05:25:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,145004877,2 - REAR-END,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",SAYLER PARK,D - DRIVER,1 - CLEAR,45233.0,03 - MID SIZE
1,9XX CHATEAU AV,39.108110,-84.560280,18-25,EAST PRICE HILL,EAST PRICE HILL,02/15/2015 03:00:00 PM,01 - NOT AN INTERSECTION,2 - INJURY,2.0,...,155002081,1 - NOT COLLISION BETWEEN TWO MOTOR VEHICLES I...,01 - DRY,2 - STRAIGHT GRADE,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,O - OCCUPANT,1 - CLEAR,45204.0,02 - COMPACT
2,30XX READING RD,39.135486,-84.496520,18-25,AVONDALE,AVONDALE,07/23/2015 11:54:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,155010090,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,O - OCCUPANT,1 - CLEAR,45206.0,04 - FULL SIZE
3,36XX READING RD,39.147889,-84.489222,61-70,AVONDALE,AVONDALE,04/21/2018 01:00:00 PM,01 - NOT AN INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,185005525,"7 - SIDESWIPE, SAME DIRECTION",01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",AVONDALE,D - DRIVER,1 - CLEAR,45229.0,07 - PICKUP
4,37XX WARSAW AV,39.110989,-84.573138,31-40,EAST PRICE HILL,EAST PRICE HILL,09/01/2018 07:59:00 PM,03 - T-INTERSECTION,3 - PROPERTY DAMAGE ONLY (PDO),3.0,...,185012267,6 - ANGLE,01 - DRY,1 - STRAIGHT LEVEL,"2 - BLACKTOP, BITUMINOUS, ASPHALT",EAST PRICE HILL,D - DRIVER,1 - CLEAR,45205.0,04 - FULL SIZE


In [8]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 258672 entries, 0 to 258671
Data columns (total 26 columns):
 #   Column                          Non-Null Count   Dtype  
---  ------                          --------------   -----  
 0   ADDRESS_X                       258669 non-null  object 
 1   LATITUDE_X                      258672 non-null  float64
 2   LONGITUDE_X                     258672 non-null  float64
 3   AGE                             258672 non-null  object 
 4   COMMUNITY_COUNCIL_NEIGHBORHOOD  253006 non-null  object 
 5   CPD_NEIGHBORHOOD                252959 non-null  object 
 6   CRASHDATE                       258669 non-null  object 
 7   CRASHLOCATION                   194021 non-null  object 
 8   CRASHSEVERITY                   258672 non-null  object 
 9   CRASHSEVERITYID                 258672 non-null  float64
 10  DATECRASHREPORTED               258670 non-null  object 
 11  DAYOFWEEK                       258671 non-null  object 
 12  GENDER          

##Интерпретация полученных признаков
| №  | Признак                                      | Тип признака       | Что это |
|----|----------------------------------------------|--------------------|--------|
| 1  | Address_X         | Текстовый      | Адрес с маскированным домом |
| 2  | COMMUNITY_COUNCIL_NEIGHBORHOOD             | Категориальный      | Район |
| 3  | CPD_NEIGHBORHOOD         | Категориальный      | Ближайшее отделение полиции |
| 4  | CPD_NEIGHBORHOOD             | Категориальный      | Тип дороги, где произошла авария |
| 5  | CRASHSEVERITY                            | Категориальный    | Летальность аварии (сразу всмятку или лайтово) |
| 6  | INSTANCEID                                      | Числовой        | ID аварии|
| 7  | LIGHTCONDITIONSPRIMARY                      | Категориальный         | Небо во время аварии |
| 8  |  MANNEROFCRASH                           | Категориальный         |  Тип аварии |
| 9 | TYPEOFPERSON                                    | Категориальный     | Кто попал в аварию (пассажир, пешеход, водитель) |
| 10 |  UNITTYPE                             | Категориальный        | Какой вид автомобиля |

In [12]:
df.CRASHSEVERITY.unique()

array(['3 - PROPERTY DAMAGE ONLY (PDO)', '2 - INJURY', '1 - FATAL INJURY',
       '5 - PROPERTY DAMAGE ONLY', '4 - INJURY POSSIBLE',
       '3 - MINOR INJURY SUSPECTED', '2 - SERIOUS INJURY SUSPECTED',
       '1 - FATAL'], dtype=object)

In [13]:
df.INJURIES.unique()

array(['1 - NO INJURY / NONE REPORTED', '3 - NON-INCAPACITATING',
       '5 - NO APPARENTY INJURY', '4 - POSSIBLE INJURY',
       '3 - SUSPECTED MINOR INJURY', '2 - POSSIBLE', '4 - INCAPACITATING',
       '2 - SUSPECTED SERIOUS INJURY', '5 - FATAL', nan, '1 - FATAL'],
      dtype=object)

In [16]:
temp = df[['INJURIES', 'CRASHSEVERITY','ADDRESS_X']].groupby(['CRASHSEVERITY','INJURIES']).count()
temp

ADDRESS_X
CRASHSEVERITY                  INJURIES                                
1 - FATAL                      1 - FATAL                             66
                               1 - NO INJURY / NONE REPORTED          3
                               2 - SUSPECTED SERIOUS INJURY          26
                               3 - SUSPECTED MINOR INJURY            32
                               4 - POSSIBLE INJURY                   13
                               5 - FATAL                              2
                               5 - NO APPARENTY INJURY               45
1 - FATAL INJURY               1 - NO INJURY / NONE REPORTED        110
                               2 - POSSIBLE                          13
                               3 - NON-INCAPACITATING                42
                               4 - INCAPACITATING                    52
                               5 - FATAL                            172
2 - INJURY                     1 - NO INJURY / NONE REPORTED      21196
                               2 - POSSIBLE                       15297
                               3 - NON-INCAPACITATING             10317
                               4 - INCAPACITATING                  2007
                               4 - POSSIBLE INJURY                    4
                               5 - NO APPARENTY INJURY                2
2 - SERIOUS INJURY SUSPECTED   1 - NO INJURY / NONE REPORTED          8
                               2 - POSSIBLE                           5
                               2 - SUSPECTED SERIOUS INJURY         486
                               3 - NON-INCAPACITATING                 4
                               3 - SUSPECTED MINOR INJURY           154
                               4 - INCAPACITATING                     8
                               4 - POSSIBLE INJURY                   56
                               5 - NO APPARENTY INJURY              303
3 - MINOR INJURY SUSPECTED     1 - NO INJURY / NONE REPORTED          9
                               2 - SUSPECTED SERIOUS INJURY           1
                               3 - NON-INCAPACITATING                 8
                               3 - SUSPECTED MINOR INJURY          4954
                               4 - POSSIBLE INJURY                  653
                               5 - NO APPARENTY INJURY             3713
3 - PROPERTY DAMAGE ONLY (PDO) 1 - NO INJURY / NONE REPORTED     144522
                               2 - POSSIBLE                           1
                               5 - NO APPARENTY INJURY               16
4 - INJURY POSSIBLE            1 - NO INJURY / NONE REPORTED          2
                               2 - POSSIBLE                           3
                               4 - POSSIBLE INJURY                 4201
                               5 - NO APPARENTY INJURY             3584
5 - PROPERTY DAMAGE ONLY       1 - NO INJURY / NONE REPORTED         64
                               4 - POSSIBLE INJURY                    1
                               5 - NO APPARENTY INJURY            46293

In [17]:
df.columns = df.columns.str.lower()

In [20]:
temp = df[['crashseverity', 'crashseverityid']].groupby(['crashseverity','crashseverityid']).count()
temp

,
crashseverity,crashseverityid
1 - FATAL,201901.0
1 - FATAL INJURY,1.0
2 - INJURY,2.0
2 - SERIOUS INJURY SUSPECTED,201902.0
3 - MINOR INJURY SUSPECTED,201903.0
3 - PROPERTY DAMAGE ONLY (PDO),3.0
4 - INJURY POSSIBLE,201904.0
5 - PROPERTY DAMAGE ONLY,201905.0


Наверное, в какой-то момент изменилось, то, как кодируют степень тяжести нанесенного ущерба. Или для разных городов разные коды

In [26]:
# Переводим в строку без '.0' (если нет пропусков) и проверяем, что длина равна 6 цифрам
df['id6figures'] = df['crashseverityid'].dropna().astype(int).astype(str).str.len() == 6

# Заполняем пропуски (если в исходной колонке были NaN) значением False
df['id6figures'] = df['id6figures'].fillna(False).astype(int)
distribution = pd.crosstab(df['crashdate'], df['id6figures'])



In [30]:
]

id6figures,0,1
crashdate,,
01/01/2013 01:04:00 AM,2,0
01/01/2013 01:31:00 PM,2,0
01/01/2013 01:35:00 AM,4,0
01/01/2013 01:45:12 AM,1,0
01/01/2013 01:57:00 AM,2,0
...,...,...
12/31/2020 11:58:00 AM,0,1
12/31/2020 11:58:00 PM,0,2
12/31/2020 12:33:00 PM,0,1


In [19]:
df.crashseverityid.unique()

array([3.00000e+00, 2.00000e+00, 1.00000e+00, 2.01905e+05, 2.01904e+05,
       2.01903e+05, 2.01902e+05, 2.01901e+05])

In [18]:
df.drop(columns = ['injuries', 'zip', 'localreportno', 'address_x',], inplace = True)

Index(['address_x', 'latitude_x', 'longitude_x', 'age',
       'community_council_neighborhood', 'cpd_neighborhood', 'crashdate',
       'crashlocation', 'crashseverity', 'crashseverityid',
       'datecrashreported', 'dayofweek', 'gender', 'injuries', 'instanceid',
       'lightconditionsprimary', 'localreportno', 'mannerofcrash',
       'roadconditionsprimary', 'roadcontour', 'roadsurface',
       'sna_neighborhood', 'typeofperson', 'weather', 'zip', 'unittype'],
      dtype='object')